# Домашнє завдання №2: Plan-and-Execute з memory та HITL

**Домен:** YouTube SMM Campaign Planner. Notebook демонструє `planner → executor → replanner`, ChromaDB, file-backed `SqliteSaver`, два HITL gates і відновлення за `thread_id`. Бізнес-логіка міститься у Python-модулях, а notebook є зручним користувацьким інтерфейсом до тієї самої реалізації.

## 1. Імпорти та конфігурація

`scripted` забезпечує відтворюваний запуск без API-ключа. Для production-демо можна замінити на `gemini`.

In [1]:
from uuid import uuid4
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, JSON

# VS Code може запускати kernel із кореня workspace, а не з папки notebook.
candidates = [
    Path.cwd(),
    Path.cwd() / 'HW2_Pylypenko_SMM_Plan_Execute',
    Path.cwd().parent / 'HW2_Pylypenko_SMM_Plan_Execute',
]
PROJECT_DIR = next((path.resolve() for path in candidates if (path / 'smm_plan_agent').is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Не знайдено папку HW2_Pylypenko_SMM_Plan_Execute зі smm_plan_agent')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from smm_plan_agent.factory import create_agent

THREAD_ID = f"notebook-{uuid4().hex[:8]}"
REQUEST = (
    "Проаналізуй YouTube тренди про AI agents за останні 7 днів, "
    "перевір brand safety та заплануй SMM-кампанію"
)
agent = create_agent(provider="scripted")
print(f"Project directory: {PROJECT_DIR}")
print(f"Thread ID: {THREAD_ID}")
print(f"Документів у ChromaDB: {agent.knowledge_base.collection.count()}")

Project directory: C:\Study\AI_agent_LLM\HW2_Pylypenko_SMM_Plan_Execute
Thread ID: notebook-8b0ad9d3
Документів у ChromaDB: 12


## 2. Запуск Plan-and-Execute

Граф сам проходить planner, три кроки executor/replanner і зупиняється на першому `interrupt()` для сумнівного контенту.

In [2]:
first_result = agent.start(REQUEST, thread_id=THREAD_ID)
state = agent.state(thread_id=THREAD_ID).values
display(Markdown(f"**Ціль:** {state['goal']}"))
display(pd.DataFrame({
    '№': range(1, len(state['plan']) + 1),
    'Крок плану': state['plan'],
    'Статус': ['виконано' if i < state['current_step'] else 'очікує' for i in range(len(state['plan']))],
}))
print('Tool history:', ' → '.join(state['tool_history']))

**Ціль:** Проаналізуй YouTube тренди про AI agents за останні 7 днів, перевір brand safety та заплануй SMM-кампанію

,№,Крок плану,Статус
0,1,Знайти свіжі YouTube-відео за темою через sear...,виконано
1,2,Отримати brand-safety правила через search_kno...,виконано
2,3,Оцінити тренди та сформувати approved і needs_...,виконано
3,4,Запланувати кампанію з фінального топу через s...,очікує


Tool history: search_recent_videos → search_knowledge → evaluate_trends


## 3. Результат Agentic RAG і два списки

`approved` уже може входити до маркетингового топу. `needs_review` не додається автоматично. `blocked` показано лише для аудиту — hard-block позиції не можна повернути звичайним review.

In [4]:
def rows(items):
    return [{
        'video_id': item['video']['video_id'],
        'title': item['video']['title'],
        'views': item['video']['views'],
        'trend_score': item['video']['trend_score'],
        'decision': item['decision'],
        'categories': ', '.join(item['categories']),
        'reason': item['reason'],
        'url': item['video']['url'],
    } for item in items]

display(Markdown('### ✅ Точно прийнятні'))
display(pd.DataFrame(rows(state['approved'])))
display(Markdown('### ⚠️ Потребують рішення людини'))
display(pd.DataFrame(rows(state['needs_review'])))
display(Markdown('### ⛔ Автоматично заблоковані'))
display(pd.DataFrame(rows(state['blocked'])))

### ✅ Точно прийнятні

,video_id,title,views,trend_score,decision,categories,reason,url
0,aiagent001,AI Agents Build a Business in 24 Hours,1850000,0.9621,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=aiagent001
1,shortsai001,AI Agent vs Traditional Automation #Shorts,1290000,0.9459,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=shortsai001
2,aicode0001,I Replaced My Workflow with Five AI Agents,1430000,0.9346,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=aicode0001
3,robot000001,This Humanoid Robot Learned a New Task,2090000,0.9262,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=robot000001
4,agentnews1,Why 2026 Is the Year of Agentic AI,760000,0.9051,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=agentnews1
5,multimodal1,Multimodal AI Agents Can Now See and Act,680000,0.8709,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=multimodal1
6,aitools0001,7 AI Tools That Actually Save Time,980000,0.8702,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=aitools0001
7,pythonag001,LangGraph ReAct Agent in Python,430000,0.8491,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=pythonag001
8,localai0001,Run an AI Agent Locally with Ollama,470000,0.8156,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=localai0001
9,ragagent001,Build a RAG Agent from Scratch,520000,0.7956,allow,,Не знайдено збігів із retrieved заборонами.,https://www.youtube.com/watch?v=ragagent001


### ⚠️ Потребують рішення людини

,video_id,title,views,trend_score,decision,categories,reason,url
0,warnewsai1,AI Explains This Week's War News,1510000,0.9420,review,sensitive_context,"Збіг із політикою: war explained, war news.",https://www.youtube.com/watch?v=warnewsai1
1,boxingai01,AI Camera Tracks Boxing Knockout Highlights,1620000,0.9395,review,sensitive_context,"Збіг із політикою: boxing knockout, fight high...",https://www.youtube.com/watch?v=boxingai01


### ⛔ Автоматично заблоковані

,video_id,title,views,trend_score,decision,categories,reason,url
0,adultbad01,Explicit Adult AI Fantasy — 18+,3100000,0.9893,block,sexual_content,"Збіг із політикою: explicit adult, nudity, por...",https://www.youtube.com/watch?v=adultbad01
1,violent001,Graphic Attack Generated by AI,2700000,0.9857,block,graphic_violence,"Збіг із політикою: blood scene, gore, graphic ...",https://www.youtube.com/watch?v=violent001


## 4. HITL №1 — рішення щодо сумнівних позицій

Змініть `acceptable_video_ids`, якщо потрібне інше рішення. Доступні також `approve_all` і `reject_all`.

In [5]:
moderation_decision = {
    'decision': 'approve_selected',
    'acceptable_video_ids': ['boxingai01'],
    'reason': 'Спортивний не-графічний контекст прийнятний; воєнні новини відхилено.',
}
second_result = agent.resume(moderation_decision, thread_id=THREAD_ID)
state = agent.state(thread_id=THREAD_ID).values
print('Наступний gate:', second_result['__interrupt__'][0].value['gate'])

Наступний gate: campaign_approval


## 5. Оновлений фінальний топ

Прийнята людиною позиція додається до існуючого топу, після чого весь список повторно сортується за `trend_score`.

In [6]:
final_rows = rows(state['final_top'])
for rank, row in enumerate(final_rows, 1):
    row['rank'] = rank
display(pd.DataFrame(final_rows)[['rank', 'video_id', 'title', 'views', 'trend_score', 'url']])
display(Markdown('### Raw action, яка очікує approval'))
display(JSON(state['pending_action']))

,rank,video_id,title,views,trend_score,url
0,1,aiagent001,AI Agents Build a Business in 24 Hours,1850000,0.9621,https://www.youtube.com/watch?v=aiagent001
1,2,shortsai001,AI Agent vs Traditional Automation #Shorts,1290000,0.9459,https://www.youtube.com/watch?v=shortsai001
2,3,boxingai01,AI Camera Tracks Boxing Knockout Highlights,1620000,0.9395,https://www.youtube.com/watch?v=boxingai01
3,4,aicode0001,I Replaced My Workflow with Five AI Agents,1430000,0.9346,https://www.youtube.com/watch?v=aicode0001
4,5,robot000001,This Humanoid Robot Learned a New Task,2090000,0.9262,https://www.youtube.com/watch?v=robot000001
5,6,agentnews1,Why 2026 Is the Year of Agentic AI,760000,0.9051,https://www.youtube.com/watch?v=agentnews1
6,7,multimodal1,Multimodal AI Agents Can Now See and Act,680000,0.8709,https://www.youtube.com/watch?v=multimodal1
7,8,aitools0001,7 AI Tools That Actually Save Time,980000,0.8702,https://www.youtube.com/watch?v=aitools0001
8,9,pythonag001,LangGraph ReAct Agent in Python,430000,0.8491,https://www.youtube.com/watch?v=pythonag001
9,10,localai0001,Run an AI Agent Locally with Ollama,470000,0.8156,https://www.youtube.com/watch?v=localai0001


### Raw action, яка очікує approval

<IPython.core.display.JSON object>

## 6. HITL №2 — approve / edit / reject ризикової дії

Нижче обрано `approve`. Для демонстрації можна поставити `reject` або `edit` з дозволеними полями `campaign_name`, `video_ids`, `planned_date`, `note`.

In [7]:
campaign_decision = {'decision': 'approve'}
final_result = agent.resume(campaign_decision, thread_id=THREAD_ID)
state = agent.state(thread_id=THREAD_ID).values
display(pd.DataFrame([{
    'thread_id': THREAD_ID,
    'completed': state['completed'],
    'campaign_status': state['campaign_status'],
    'steps_completed': state['current_step'],
    'replans': state['replan_count'],
    'final_top_size': len(state['final_top']),
}]))
print('Tool history:', ' → '.join(state['tool_history']))

,thread_id,completed,campaign_status,steps_completed,replans,final_top_size
0,notebook-865523ef,True,scheduled,4,0,10


Tool history: search_recent_videos → search_knowledge → evaluate_trends → schedule_campaign → schedule_campaign:executed


## 7. Persistence після повторного створення агента

Закриваємо SQLite connection, створюємо новий об'єкт і читаємо стан за тим самим `thread_id`. Це імітує restart процесу.

In [8]:
expected_status = state['campaign_status']
expected_step = state['current_step']
agent.close()

restored_agent = create_agent(provider='scripted')
restored = restored_agent.state(thread_id=THREAD_ID).values
display(pd.DataFrame([{
    'same_thread_id': THREAD_ID,
    'restored_step': restored['current_step'],
    'step_matches': restored['current_step'] == expected_step,
    'restored_campaign_status': restored['campaign_status'],
    'status_matches': restored['campaign_status'] == expected_status,
}]))

,same_thread_id,restored_step,step_matches,restored_campaign_status,status_matches
0,notebook-865523ef,4,True,scheduled,True


## 8. Незалежність thread_id

Новий thread отримує власний план і checkpoint та не змінює завершений workflow.

In [9]:
OTHER_THREAD_ID = f"notebook-other-{uuid4().hex[:8]}"
other_result = restored_agent.start(REQUEST, thread_id=OTHER_THREAD_ID)
other_state = restored_agent.state(thread_id=OTHER_THREAD_ID).values
original_state = restored_agent.state(thread_id=THREAD_ID).values
display(pd.DataFrame([
    {'thread_id': THREAD_ID, 'step': original_state['current_step'], 'status': original_state['campaign_status']},
    {'thread_id': OTHER_THREAD_ID, 'step': other_state['current_step'], 'status': other_state['campaign_status']},
]))
restored_agent.close()

,thread_id,step,status
0,notebook-865523ef,4,scheduled
1,notebook-other-3f3baa5d,3,not_requested


## Висновок

Продемонстровано всі критерії ДЗ: structured `Plan` і `ReplanDecision`, покроковий executor, file-backed SqliteSaver, незалежні threads, ChromaDB із 12 документами, самостійний вибір RAG tool, два HITL interrupts і фактичне виконання ризикового tool лише після approval.